# Prepare Atomic Files

Loads the filtered CSV from `process-jsonl.ipynb` and converts it into RecBole Atomic Files.
Run `process-jsonl.ipynb` first to generate `data/processed/reviews.csv.gz`.

In [1]:
from pathlib import Path
import pandas as pd

In [18]:
# --- Config ---
TARGET_CATEGORY = "Beauty_and_Personal_Care"
DATA_DIR: str = "../data"

# Data split cutoff dates
TRAIN_END_CUTOFF_DATE: str = "2022-08-01"
VALID_END_CUTOFF_DATE: str = "2022-10-01"

# User review count thresholds
USER_MIN_REVIEWS: int = 5
WARM_USER_MIN_REVIEWS: int = 10

# Downsampling for faster iteration
RANDOM_SEED: int = 42
MAX_TRAIN_SIZE: int | None = None
MAX_VALID_SIZE: int | None = None
MAX_TEST_SIZE: int | None = None

## Load reviews

In [3]:
df_reviews = pd.read_csv(f"{DATA_DIR}/reviews.csv")
display(df_reviews.sample(10))
df_reviews.info()

,user_id,parent_asin,rating,timestamp,category
5138272,AGYOGNAKXI7U4LC4LYOYMGHUQ7UA,B07D7PWMQ4,1.0,1619716654036,Clothing_Shoes_and_Jewelry
9813841,AEO7NFHTQSZZK5VPNS3Q2X4WI6KQ,B08CXX1Z1Q,4.0,1629725284890,Clothing_Shoes_and_Jewelry
10991790,AHOHNPCUN4KKCXV6QDW2T7RVHTYA,B07559KX9D,5.0,1632966909155,Clothing_Shoes_and_Jewelry
19351718,AEGGFR77HLDKARU35M4EL6FFDCPA,B07KTLS16H,5.0,1655165204592,Clothing_Shoes_and_Jewelry
19647697,AGV2YW5B3SCKG6WV422CIITZOGRQ,B09P52CLL9,5.0,1655903660238,Clothing_Shoes_and_Jewelry
13460001,AESX3TKSL2AGNMETSH6KXNCMYQ7Q,B089Y8FTRG,5.0,1639952399392,Beauty_and_Personal_Care
21525865,AG7IELMW3LS36FMZGL2YSCVI25TQ,B07MC3YFXX,1.0,1660175041593,Clothing_Shoes_and_Jewelry
3937112,AES4YGDWUDR7RWPAVBFISNIINSCQ,B08VB3HMSB,5.0,1617480380618,Clothing_Shoes_and_Jewelry
19260967,AEIF5Q7PZUJA2YUM4AV2X4N3EMMA,B09TTYMJ9R,2.0,1654913517608,Clothing_Shoes_and_Jewelry
22487463,AEW6QZRGTEL7T55CCLMNRLDWF5OQ,B07QB4QLJ7,5.0,1662398744509,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26876611 entries, 0 to 26876610
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      object 
 1   parent_asin  object 
 2   rating       float64
 3   timestamp    int64  
 4   category     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 1.0+ GB


## Load items

In [4]:
df_items = pd.read_csv(f"{DATA_DIR}/items.csv")
display(df_items.sample(10))
df_items.info()

,parent_asin,title,price,store,category
936324,B09LHQXT5F,Funlingo Women's Tankini Swimsuits Two Piece B...,NaN,Funlingo,Clothing_Shoes_and_Jewelry
1234521,B01FYEH1XK,Bruno Marc Men's Oxford Dress Shoes Lace up Fo...,NaN,Bruno Marc,Clothing_Shoes_and_Jewelry
2285119,B09HZX5CFM,YESNO Women V Neck Tunic Waffle Knit Shirt Lon...,NaN,YESNO,Clothing_Shoes_and_Jewelry
1308846,B09HR41S4Q,Hilary Radley Women's Stripes Bermuda Short (O...,19.69,Hilary,Clothing_Shoes_and_Jewelry
574374,B07NPT5JB4,Marvel Avengers Boys Raglan T-Shirt & Shorts S...,16.99,Marvel,Clothing_Shoes_and_Jewelry
2128635,B01MZDQE1Y,Messy Bun Hat Beanie (Blue) CC Quality Knit,NaN,BeYOUtiful Trends,Clothing_Shoes_and_Jewelry
70573,B081CM1CF8,Fiezkaa 110 Pairs Eyelash Extension Pads - Las...,NaN,fiezkaa,Beauty_and_Personal_Care
1264800,B07YM4ZMF3,SOHO GLAM High Waisted Stretchy Elastic Bell B...,NaN,SOHO GLAM,Clothing_Shoes_and_Jewelry
598753,B07BFJPB25,Cielo Women's Solid Basic Open Front Pockets K...,22.13,Cielo,Clothing_Shoes_and_Jewelry
1400413,B097SVV9N2,JVenus Ladies underwear Lace babydoll pajamas ...,14.99,JVenus,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2694121 entries, 0 to 2694120
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   parent_asin  object 
 1   title        object 
 2   price        float64
 3   store        object 
 4   category     object 
dtypes: float64(1), object(4)
memory usage: 102.8+ MB


## Map user/item IDs to integers

In [19]:
# As RecBole expects integer IDs, we need to create mappings from the original string IDs to integers.
user_ids: set[str] = set(df_reviews["user_id"])
item_ids: set[str] = set(df_reviews["parent_asin"])

user_map: dict[str, int] = {uid: i+1 for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i+1 for i, pid in enumerate(sorted(item_ids))}
print(f"For all categories, Users: {len(user_map):,}  Items: {len(item_map):,}")

For all categories, Users: 11,700,249  Items: 2,694,121


In [20]:
df_reviews['uid'] = df_reviews['user_id'].map(user_map)
df_reviews['iid'] = df_reviews['parent_asin'].map(item_map)
df_items['iid'] = df_items['parent_asin'].map(item_map)

## Split train/valid/test with cutoff timestamps

In [21]:
# Only reviews from the target category
df_target_reviews = df_reviews[df_reviews["category"] == TARGET_CATEGORY]
df_target_reviews

,user_id,parent_asin,rating,timestamp,category,uid,iid
0,AHT4VFETXXM7EXTAEEDNI5CWUCCQ,B078SB5PHG,3.0,1609459200378,Beauty_and_Personal_Care,11068386,390262
7,AHVNPQDHZJMTNY6JFTMWR6PC5N5A,B08HCSC55R,3.0,1609459207570,Beauty_and_Personal_Care,11299045,1336160
10,AG7Q6476PIPY2STEB2XSWZ2PVTTA,B09N3PZH17,5.0,1609459210794,Beauty_and_Personal_Care,6370512,2232414
17,AFXBP2X6W3CPOKKZE6WZAECUTDUQ,B006988GRQ,2.0,1609459216747,Beauty_and_Personal_Care,5598738,48046
21,AHSRIBJ5QM2LCTXVUVDUFDYXSPLA,B07B53Q3BX,4.0,1609459224515,Beauty_and_Personal_Care,11035860,413067
...,...,...,...,...,...,...,...
26876594,AE4GNI3M4AUQZZRK3VGFEUQ2WTIQ,B07GL7H9B8,5.0,1672444781572,Beauty_and_Personal_Care,219111,517624
26876599,AGMJEEBDSURZGOSMADCYVYQOYS2Q,B01N2RIEJ9,5.0,1672444786916,Beauty_and_Personal_Care,7539174,259815
26876605,AHA6W3QHAPWFCNN5WIUM4N36NXLQ,B08WJDVC2T,5.0,1672444794019,Beauty_and_Personal_Care,9336107,1679335
26876607,AFYNEQAMDBWVGYSXYSXLALWPWN5A,B0BJLPRXZ6,1.0,1672444797860,Beauty_and_Personal_Care,5723253,2599551


In [22]:
# Filter users with fewer than USER_MIN_REVIEWS reviews
before_users = df_target_reviews["uid"].nunique()
user_counts = df_target_reviews.groupby("uid").size()
valid_users = user_counts[user_counts >= USER_MIN_REVIEWS].index
df_target_reviews = df_target_reviews[df_target_reviews["uid"].isin(valid_users)]
after_users = df_target_reviews["uid"].nunique()
removed_users = before_users - after_users
removed_reviews = len(df_target_reviews) - df_target_reviews["uid"].isin(valid_users).sum()
print(f"Removed {removed_users:,} users with < {USER_MIN_REVIEWS} reviews "
      f"({after_users:,} users remain, {len(df_target_reviews) - (before_users - removed_users):,} reviews removed)")

Removed 4,221,111 users with < 5 reviews (168,377 users remain, 1,417,528 reviews removed)


In [23]:
# Determine cutoff timestamps for train/valid/test splits based on fixed dates
def _date_to_ms(date_str: str) -> int:
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

timestamps = sorted(df_target_reviews["timestamp"].values)
train_start_ts = timestamps[0]
train_end_ts = _date_to_ms(TRAIN_END_CUTOFF_DATE)
valid_end_ts = _date_to_ms(VALID_END_CUTOFF_DATE)
print(f"Train start timestamp: {train_start_ts} ({pd.Timestamp(train_start_ts, unit='ms', tz='UTC')})")
print(f"Train end timestamp: {train_end_ts} ({pd.Timestamp(train_end_ts, unit='ms', tz='UTC')})")
print(f"Valid end timestamp: {valid_end_ts} ({pd.Timestamp(valid_end_ts, unit='ms', tz='UTC')})")

Train start timestamp: 1609459264769 (2021-01-01 00:01:04.769000+00:00)
Train end timestamp: 1659312000000 (2022-08-01 00:00:00+00:00)
Valid end timestamp: 1664582400000 (2022-10-01 00:00:00+00:00)


## Split train/valid/test for target category

In [24]:
df_target_train = df_target_reviews[df_target_reviews["timestamp"] <= train_end_ts]
df_target_valid = df_target_reviews[(df_target_reviews["timestamp"] > train_end_ts) & (df_target_reviews["timestamp"] <= valid_end_ts)]
df_target_test = df_target_reviews[df_target_reviews["timestamp"] > valid_end_ts]
print(f"Target train reviews: {len(df_target_train)}")
print(f"Target valid reviews: {len(df_target_valid)}")
print(f"Target test reviews: {len(df_target_test)}")

Target train reviews: 1125023
Target valid reviews: 168239
Target test reviews: 292643


## Define cold vs. warm users with train data in target category

In [25]:
df_train_users = df_target_train.groupby("uid").size().reset_index(name="num_train")
cold_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] < WARM_USER_MIN_REVIEWS]["uid"])
warm_user_ids: set[str] = set(df_train_users[df_train_users["num_train"] >= WARM_USER_MIN_REVIEWS]["uid"])
print(f"Warm users (>= {WARM_USER_MIN_REVIEWS} reviews): {len(warm_user_ids)}")
print(f"Cold users (< {WARM_USER_MIN_REVIEWS} reviews): {len(cold_user_ids)}")

Warm users (>= 10 reviews): 20100
Cold users (< 10 reviews): 136387


## Downsample the train/valid/test sets

In [26]:
if MAX_TRAIN_SIZE is not None and len(df_target_train) > MAX_TRAIN_SIZE:
    df_target_train = df_target_train.sample(n=MAX_TRAIN_SIZE, random_state=RANDOM_SEED)
if MAX_VALID_SIZE is not None and len(df_target_valid) > MAX_VALID_SIZE:
    df_target_valid = df_target_valid.sample(n=MAX_VALID_SIZE, random_state=RANDOM_SEED)
if MAX_TEST_SIZE is not None and len(df_target_test) > MAX_TEST_SIZE:
    df_target_test = df_target_test.sample(n=MAX_TEST_SIZE, random_state=RANDOM_SEED)


print(f"After downsampling (if applicable):")
print(f"Target train reviews: {len(df_target_train)}")
print(f"Target valid reviews: {len(df_target_valid)}")
print(f"Target test reviews: {len(df_target_test)}")

train_users = set(df_target_train["uid"].unique())
df_target_valid = df_target_valid[df_target_valid["uid"].isin(train_users)]
df_target_test = df_target_test[df_target_test["uid"].isin(train_users)]
print(f"After filtering out users with no training history:")
print(f"Target valid reviews: {len(df_target_valid)}")
print(f"Target test reviews: {len(df_target_test)}")

df_all_target = pd.concat([df_target_train, df_target_valid, df_target_test])
print('Unique users:', len(df_all_target['uid'].unique()))
print('Unique items:', len(df_all_target['iid'].unique()))
print('Train interactions per user', len(df_target_train) / len(df_target_train['uid'].unique()))
df_all_target

After downsampling (if applicable):
Target train reviews: 1125023
Target valid reviews: 168239
Target test reviews: 292643
After filtering out users with no training history:
Target valid reviews: 139338
Target test reviews: 197706
Unique users: 156487
Unique items: 250852
Train interactions per user 7.18924255688971


,user_id,parent_asin,rating,timestamp,category,uid,iid
43,AE4RJF4WSFBAT3PT5ZF4HOANBMMA,B09GN3GC8C,5.0,1609459264769,Beauty_and_Personal_Care,250513,2117925
46,AHPUMK34NKI3FO7UWN3RZ5ZZBXLA,B079ZC5XSP,5.0,1609459267632,Beauty_and_Personal_Care,10770822,410109
64,AHTDYGXHONM2BHENDRMKMC34ZZZA,B0831MY9VW,5.0,1609459303648,Beauty_and_Personal_Care,11088513,974774
104,AEUP74AWAO5S27N7SBH5BNSHKIHA,B0BWNHGP2Q,3.0,1609459392279,Beauty_and_Personal_Care,2438420,2644606
115,AE5XXWK5FGU62ROWKSF2VR2EAI7Q,B086P9GHR1,5.0,1609459406189,Beauty_and_Personal_Care,360957,1068034
...,...,...,...,...,...,...,...
26876267,AGLE62AMIJO4M4FFGFGRML5A5XFA,B0BW4XM6WP,5.0,1672444369904,Beauty_and_Personal_Care,7432658,2642797
26876301,AF76BUVOIWP2QIIQSKUET7LZHMKQ,B071SDJTNH,3.0,1672444422571,Beauty_and_Personal_Care,3395787,300446
26876372,AGLE62AMIJO4M4FFGFGRML5A5XFA,B00NXRC7DU,5.0,1672444504136,Beauty_and_Personal_Care,7432658,120987
26876441,AGVR7COFPX7ZENTTVJ2UYDBAIJBQ,B0BX5MM7F1,5.0,1672444602726,Beauty_and_Personal_Care,8382879,2646217


## Write atomic files

In [29]:
def get_user_category(uid: str) -> int:
    if uid in warm_user_ids:
        return 0  # Warm user
    return 1      # Cold user
    
def write_user_file(path: Path, uids: set[str]) -> None:
    rows = [(uid, get_user_category(uid)) for uid in uids]
    df_out = pd.DataFrame(rows, columns=["user_id:token", "category:token"])
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_item_file(path: Path, df: pd.DataFrame) -> None:
    df_out = df[["iid", "store", "price"]].copy()
    df_out["store"] = df_out["store"].fillna("").astype(str).str.replace('"', "", regex=False)
    df_out["price"] = pd.to_numeric(df_out["price"], errors="coerce").fillna("")
    df_out.columns = ["item_id:token", "store:token", "price:float"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")

def write_inter_file(path: Path, df: pd.DataFrame) -> None:
    df_out = df[["uid", "iid", "rating", "timestamp"]].copy()
    df_out.columns = ["user_id:token", "item_id:token", "rating:float", "timestamp:float"]
    df_out.to_csv(path, sep="\t", index=False)
    print(f"Wrote {path} ({len(df_out):,} rows)")


In [30]:
# Write RecBole atomic files for the target category
target_dataset_prefix = Path(DATA_DIR) / "target" / "target"
target_dataset_prefix.parent.mkdir(parents=True, exist_ok=True)
write_inter_file(target_dataset_prefix.with_suffix(".train.inter"), df_target_train)
write_inter_file(target_dataset_prefix.with_suffix(".valid.inter"), df_target_valid)
write_inter_file(target_dataset_prefix.with_suffix(".test.inter"), df_target_test)
write_user_file(target_dataset_prefix.with_suffix(".user"), df_all_target['uid'].unique())
write_item_file(target_dataset_prefix.with_suffix(".item"), df_items[df_items['iid'].isin(df_all_target['iid'].unique())])

Wrote ../data/target/target.train.inter (1,125,023 rows)
Wrote ../data/target/target.valid.inter (139,338 rows)
Wrote ../data/target/target.test.inter (197,706 rows)
Wrote ../data/target/target.user (156,487 rows)
Wrote ../data/target/target.item (250,852 rows)
